In [ ]:
%pip install rtree

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:

import os
import json
import xml.etree.ElementTree as ET
import sumolib
import math
import matplotlib.pyplot as plt
from sumolib.net import Net
import sys
import pandas as pd
import traci           


 La fonction suivante prend en entrée un fichier JSON contenant les informations de la RSU et en extrait sa latitude, sa longitude et son orientation.

In [ ]:
def extract_lat_lon_heading(rsu_file_path):
    try:
        with open(rsu_file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
            geographical_position = data.get("geographicalPosition", {})
            latitude = geographical_position.get("latitude")
            longitude = geographical_position.get("longitude")
            heading = data.get("trueHeading")

            if latitude is not None and longitude is not None and heading is not None:
                return latitude, longitude,heading
            else:
                raise ValueError(
                    "Latitude or Longitude not found in JSON file.")       
    except (json.JSONDecodeError, FileNotFoundError, ValueError) as e:
        print(f"Error: {e}")
        return None

La fonction suivante récupère les noms des fichiers contenant des informations sur les objets capturés (les fichiers ayant l'extension `.csv`).

In [4]:
def retrieve_csv_files(directory):
    try:
        csv_files = sorted([file for file in os.listdir(directory) if file.endswith(".csv")])
        return csv_files
    except FileNotFoundError:
        print(f"The directory '{directory}' does not exist.")

Cette fonction transforme des coordonnées relatives par rapport à un RSU en une position GPS absolue.

In [5]:
def map_object(RSULat,RSULon,heading,positionX, positionY):
    # Convert heading to radians
    calibAngle = heading * math.pi / 180
    r_earth = 6378137  # Earth's radius in meters

    # Transform coordinates
    yrel = float(positionX)
    xrel = float(positionY)
    
    mod = math.sqrt(xrel*xrel + yrel*yrel)
    angle = math.atan2(yrel, xrel)
    angle -= calibAngle
    xrel = mod * math.cos(angle)
    yrel = mod * math.sin(angle)

    # Compute new GPS coordinates
    dLat = (yrel / r_earth) * (180 / math.pi)
    dLon = (xrel / (r_earth * math.cos(RSULat * math.pi / 180))) * (180 / math.pi)
    objLat = RSULat + dLat
    objLon = RSULon + dLon

    return objLat, objLon

Cette fonction permet de reclassifier les objets marqués comme "UNKNOWN" en fonction des classes issues d’une autre perception du même objet.

In [ ]:
def classify_objects():
    dataset_dir = "dataset1"
    output_dir = "dataset1_classified"
    threshold = 70.0
    os.makedirs(output_dir, exist_ok=True)

    csv_files = retrieve_csv_files(dataset_dir)
    all_rows = []
    ids_classes = {}

    # Step 1: Read and merge all data
    for csv_file in csv_files:
        input_file = os.path.join(dataset_dir, csv_file)
        df = pd.read_csv(input_file, delimiter=';', encoding='utf-8')
        all_rows.append(df)

    all_data = pd.concat(all_rows, ignore_index=True)

    base_timestamp = int(all_data.iloc[0]['timestamp'])  
    # Step 2: Determine dominant class for each object ID
    unique_ids = all_data['Id'].unique()
    for object_id in unique_ids:
        object_df = all_data[all_data['Id'] == object_id]
        vehicle_df = object_df[object_df['Class'] == 'VEHICLE']
        human_df = object_df[object_df['Class'] == 'HUMAN']
        count = object_df.shape[0]

        vehicle_proportion = (vehicle_df.shape[0] / count) * 100
        human_proportion = (human_df.shape[0] / count) * 100

        if human_proportion == 0.0 and vehicle_proportion > 0.0:
            ids_classes[object_id] = "VEHICLE"
        elif vehicle_proportion == 0.0 and human_proportion > 0.0:
            ids_classes[object_id] = "HUMAN"
        elif vehicle_proportion > threshold :
            ids_classes[object_id] = "VEHICLE"
        elif human_proportion > threshold:
            ids_classes[object_id] = "HUMAN"
        # Else: Do not include this object_id at all

    # Step 3: Filter and update class values per file, then save
    for csv_file in csv_files:
        input_file = os.path.join(dataset_dir, csv_file)
        df = pd.read_csv(input_file, delimiter=';', encoding='utf-8')

        # Keep only rows with IDs that had a dominant class
        df = df[df['Id'].isin(ids_classes.keys())]

        # Update class values to the dominant one
        df['Class'] = df['Id'].map(ids_classes)

        output_file = os.path.join(output_dir, csv_file)
        df.to_csv(output_file, sep=';', index=False, encoding='utf-8')

    print(f"Classification complete. Files saved to '{output_dir}'")
    return base_timestamp

Cette fonction parcourt les fichiers CSV et retourne, pour chaque objet, l'ensemble des points qu’il a traversés.

In [7]:
def process_data(origin, netfile, object_class):
    try:
        dataset_dir = "dataset1_classified"
        objects_data = {}
        objects_timestamps = {}
        first_timestamp = None
        
        csv_files = retrieve_csv_files(dataset_dir)
        net = sumolib.net.readNet(netfile)
        
        filtered_rows = []
        
        # Read all CSV data into memory using pandas
        for csv_file in csv_files:
            input_file = os.path.join(dataset_dir, csv_file)
            df = pd.read_csv(input_file, delimiter=';', encoding='utf-8')

            filtered_rows.append(df[df['Class'] == object_class])
        
        filtered_data_df = pd.concat(filtered_rows, ignore_index=True)
        
        # Process each unique vehicle ID from vehicle_rows
        for _ ,object in filtered_data_df.iterrows():
            id = object['Id']
            timestamp = float(object['timestamp'])
            position_x = object['positionX']
            position_y = object['positionY']
            if first_timestamp is None:
                first_timestamp = timestamp

            if id not in objects_data:
                adjusted_timestamp = (timestamp - first_timestamp) / 1000.0
                objects_timestamps[id] = adjusted_timestamp
                objects_data[id] = []
            # Adjust positions based on origin
            object_lat, object_lon = map_object(
                    origin[0], origin[1], origin[2], float(position_x), float(position_y)
            )
            adjusted_x, adjusted_y = net.convertLonLat2XY(object_lon, object_lat)
                
            # Append adjusted positions to the vehicle's data
            objects_data[id].append((adjusted_x, adjusted_y))
        
        return objects_data, list(objects_timestamps.values())

    except Exception as e:
        print(f"Error processing file: {e}")


Cette fonction génère osm.passenger.rou.xml, qui contient les routes pour les véhicules.

In [ ]:
def generate_vehicle_routes(json_file, net_file):
    output_file = "osm.passenger.rou.xml"
    rsu_location = extract_lat_lon_heading(json_file)
    if rsu_location is None:
        print("Error: Unable to extract latitude, longitude and heading")
        return
    
   
    vehicle_data,vehicle_timestamps= process_data(rsu_location,net_file,'VEHICLE')
    net = sumolib.net.readNet(net_file)
    root = ET.Element("routes", {
        "xmlns:xsi": "http://www.w3.org/2001/XMLSchema-instance",
        "xsi:noNamespaceSchemaLocation": "http://sumo.dlr.de/xsd/routes_file.xsd"
    })
    
    ET.SubElement(root, "vType", {"id": "veh_passenger", "vClass": "passenger"})
    tid = 0
   
    for trace in vehicle_data.values():
        edges = [e.getID() for e in sumolib.route.mapTrace(trace, net, 10.0,"passenger") if e.getFunction() != "internal"]
        edges = [edge for i, edge in enumerate(edges) if i == 0 or edge != edges[i-1]]
        
        if edges:
            vehicle = ET.SubElement(root, "vehicle", {
                "id": f"veh{tid}",
                "type": "veh_passenger",
                "depart": str(round(vehicle_timestamps[tid],2)),
                "departLane": "best"
            })
            ET.SubElement(vehicle, "route", {"edges": " ".join(edges)})
        else:
            print(f"No edges are found for {tid}.")
        tid+=1
    ET.indent(root)
    tree = ET.ElementTree(root)
    with open(output_file, "wb") as file:
        tree.write(file, encoding="UTF-8", xml_declaration=True)
    
    print(f"Vehicle route file created: {output_file}")


Cette fonction génère osm.pedestrian.rou.xml, qui contient les routes pour les piétons.

In [ ]:
def generate_pedestrian_paths(json_file, net_file):
    output_file = "osm.pedestrian.rou.xml"
    rsu_location = extract_lat_lon_heading(json_file)
    if rsu_location is None:
        print("Error: Unable to extract latitude, longitude and heading")
        return
    
   
    human_data,human_timestamps= process_data(rsu_location,net_file,'HUMAN')
    net :Net = sumolib.net.readNet(net_file)
    root = ET.Element("routes", {
        "xmlns:xsi": "http://www.w3.org/2001/XMLSchema-instance",
        "xsi:noNamespaceSchemaLocation": "http://sumo.dlr.de/xsd/routes_file.xsd"
    })
    
    ET.SubElement(root, "vType", {"id": "ped_pedestrian", "vClass": "pedestrian"})
    tid = 0
   
    for trace in human_data.values():
        edges = [e.getID() for e in sumolib.route.mapTrace(trace, net, 10.0,"pedestrian") if e.getFunction() == ""]
        edges = [edge for i, edge in enumerate(edges) if i == 0 or edge != edges[i-1]]
        
        if edges:
            person = ET.SubElement(root, "person", {
                "id": f"ped{tid}",
                "type": "ped_pedestrian",
                "depart": str(round(human_timestamps[tid],2)),
            })
            ET.SubElement(person, "walk", {"edges": " ".join(edges)})
        else:
            print(f"No edges are found for {tid}.")
        tid+=1
    ET.indent(root)
    tree = ET.ElementTree(root)
    with open(output_file, "wb") as file:
        tree.write(file, encoding="UTF-8", xml_declaration=True)
    
    print(f"Pedestrians file created: {output_file}")

Cette fonction génère un fichier traces.csv où les positions sont corrigées.

In [ ]:
def traci_CSV(json_file, net_file):
    try:
        dataset_dir = "dataset1_classified"
        rows = []
        net = sumolib.net.readNet(net_file)
        csv_files = retrieve_csv_files(dataset_dir)
        rsu_location = extract_lat_lon_heading(json_file)
        traces = []

        for csv_file in csv_files:
            input_file = os.path.join(dataset_dir, csv_file)
            df = pd.read_csv(input_file, delimiter=';', encoding='utf-8')
            rows.append(df)

        data_df = pd.concat(rows, ignore_index=True)
        for _ , row in data_df.iterrows():
                Lat, Lon = map_object(
                    rsu_location[0],
                    rsu_location[1],
                    rsu_location[2],
                    row["positionX"],
                    row["positionY"]
                )
                x, y = net.convertLonLat2XY(Lon, Lat)
                id = row['Id']
                speed = float(row['Vel']) / 3.6 
                object_class = row['Class']
                timestamp = row['timestamp']
                traces.append((id, x, y, speed,timestamp,object_class))
                
        # Sauvegarde dans un fichier CSV
        trace_df = pd.DataFrame(traces, columns=["Id", "X", "Y", "Speed","timestamp", "Class"])
        trace_df.to_csv("trace.csv", index=False)

    except Exception as e:
        print(f"Error processing file: {e}")

Cette fonction retourne des intervalles où la simulation se déroule sans problème, afin de pouvoir diviser le scénario en chunks.

In [ ]:

def generate_simulation_intervals(BASE_TIMESTAMP, net_file):
    if "SUMO_HOME" in os.environ:
        sys.path.append(os.path.join(os.environ["SUMO_HOME"],"tools"))
        
    trace_file = "trace.csv"
    df = pd.read_csv(trace_file, delimiter=',', encoding='utf-8')
    offset = 0
    current_time = None
    with open("seconds.txt","w") as f:
        while True:
            try :
                # We relaunch the simulation after the moment the error occured  
                # BASE_TIMESTAMP is the first timestamp of the simulation
                df = df[df["timestamp"] >= BASE_TIMESTAMP + offset*1000]
                df = df.sort_values("timestamp")
                grouped = df.groupby("timestamp")

                sumoBinary = sumolib.checkBinary('sumo')
                traci.start([
                    sumoBinary,
                    "-n", net_file,  
                ])

                traci.route.add("trip", ["1151278799"])

                first_timestamp = None 

                for timestamp, group in grouped:
                    current_timestamp = int(timestamp)
                    
                    if first_timestamp is None:
                        first_timestamp = current_timestamp
                    
                    current_time = (current_timestamp - first_timestamp) / 1000
                    
                    traci.simulationStep(current_time)
                    
                    current_vehicles = list(traci.vehicle.getIDList())
                    current_persons = list(traci.person.getIDList())
                    
                    perceived_vehicles = []
                    perceived_persons = []

                    for _, row in group.iterrows():

                        object_id = str(row['Id'])         
                        x = float(row['X'])              
                        y = float(row['Y'])                 
                        speed = float(row['Speed'])       
                        object_class = row['Class']      

                        if object_id == "9906507":
                            continue
                        if object_class == "VEHICLE":
                            perceived_vehicles.append(object_id)
                            
                            if object_id not in current_vehicles:
                                traci.vehicle.add(vehID=object_id, routeID="trip")
                            
                            traci.vehicle.moveToXY(vehID=object_id, edgeID="-1", laneIndex=-1, 
                                                x=x, y=y, keepRoute=0)
                            
                            traci.vehicle.setSpeed(vehID=object_id, speed=speed)
                        
                        else: 
                            perceived_persons.append(object_id)
                            
                            pos = traci.simulation.convertRoad(x, y)

                            if pos[0].startswith(":"):
                                continue
                            
                            if object_id not in current_persons:
                                traci.person.add(
                                    personID=object_id,
                                    edgeID="184963264#1",           
                                    pos=0,                         
                                    depart=math.ceil(current_time)
                                )
                            
                            traci.person.moveToXY(personID=object_id, edgeID=pos[0], 
                                                x=x, y=y, keepRoute=0)
                            
                            traci.person.setSpeed(personID=object_id, speed=speed)

                    vehicles_to_remove = [veh_id for veh_id in current_vehicles 
                                        if veh_id not in perceived_vehicles]
                    
                    persons_to_remove = [p_id for p_id in current_persons 
                                        if p_id not in perceived_persons]
                    
                    for veh_id in vehicles_to_remove:
                        traci.vehicle.remove(vehID=veh_id)                                
                    
                    for p_id in persons_to_remove:
                        traci.person.remove(personID=p_id)
                print("Simulation terminee avec succes")
                f.write(f"{offset},{int(current_time)}\n")
                traci.close()
                break
            except Exception as e:
                # if there is a problem, we store the instant where the error happened and store in a file
                # the time interval where the simulation run without problems
                print(e)
                stop = offset + ((current_time -100) // 100) * 100
                # We store only chunks with duration > 1000 seconds
                if stop-offset >= 1000:
                    f.write(f"{offset},{stop}\n")
                    f.flush()  
                # The simulation restarts after the instant when the error happened 
                offset = offset + math.ceil((current_time+100) / 100) *100
                traci.close()

Cette fonction transforme les intervalles en secondes en intervalles en timestamps.

In [ ]:
def seconds_to_timestamps(BASE_TIMESTAMP):
    input_file = "seconds.txt"
    output_file = "timestamps.txt"

    with open(input_file, "r") as infile, open(output_file, "w") as outfile:
        for line in infile:
            line = line.strip()
            if not line:  # skip empty lines
                continue

            try:
                # Split into two integers
                num1_str, num2_str = line.split(",")
                num1, num2 = int(num1_str.strip()), int(num2_str.strip())

                # Convert to timestamps
                ts1 = num1 * 1000 + BASE_TIMESTAMP
                ts2 = num2 * 1000 + BASE_TIMESTAMP

                # Write to output file
                outfile.write(f"{ts1},{ts2}\n")

            except ValueError:
                print(f"Skipping invalid line: {line}")


Cette fonction divise le jeu de données en chunks en se basant sur les intervalles générés précédemment.

In [11]:
def generate_chuncks():
    TRACE_FILE = 'trace.csv'
    SECONDS_FILE = 'timestamps.txt'
    OUTPUT_DIR = 'output'

    # Create output directory if it doesn't exist
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Load the trace CSV
    df = pd.read_csv(TRACE_FILE)

    # Read and parse seconds.txt
    with open(SECONDS_FILE, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    # Process each line in seconds.txt
    for i, line in enumerate(lines):
        parts = line.split(',')
        parts = [p for p in parts if p]  # remove empty strings

        if len(parts) == 2:
            start_ts = int(parts[0]) 
            end_ts = int(parts[1]) 
            mask = (df['timestamp'] >= start_ts) & (df['timestamp'] <= end_ts)
        else:
            continue  # skip malformed lines

        # Filter and save the corresponding data
        filtered_df = df[mask]
        output_path = os.path.join(OUTPUT_DIR, f"trace_{i}.csv")
        filtered_df.to_csv(output_path, index=False)

    print(f"Finished splitting into {len(lines)} files in '{OUTPUT_DIR}' directory.")


Première méthode : générer les routes via des fichiers de configuration.

In [ ]:
json_file = "rsu1.json"
net_file = "osm.net.xml.gz"
classify_objects()
generate_vehicle_routes(json_file, net_file)
generate_pedestrian_paths(json_file, net_file)

Deuxième méthode : générer les traces pour TraCI.

In [ ]:
json_file = "rsu1.json"
net_file = "osm.net.xml.gz"
# Classifier les objets
base_timestamp= classify_objects()
# Générer les traces pour TraCI avec les positions corrigées
traci_CSV(json_file,net_file)
# Identifier les intervalles où la simulation se déroule sans problème
generate_simulation_intervals(base_timestamp,net_file)
# Transformer les intervalles en secondes en intervalles en timestamps.
seconds_to_timestamps(base_timestamp)
# Diviser les traces en plusieurs fichiers en fonction des intervalles générés
generate_chuncks()
